In [ ]:
import pandas as pd
import json
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
from tqdm import tqdm
import os
from math import radians, sin, cos, sqrt, atan2
from concurrent.futures import ThreadPoolExecutor, as_completed
from functools import lru_cache

# ============================
# CONFIG
# ============================
BMKG_PATH = r"D:\indonesia_output\BMKG_Earthquake_Catalog.csv"
USGS_PATH = r"D:\indonesia_output\USGS_Earthquake_Catalog.csv"
OUTPUT_JSON = r"D:\indonesia_output\radar_merged_bmkg_usgs_2001_2024_promax.json"

STATION_CODES = [
    "BJI", "BBJI", "GSI", "KAPI", "LBMI", "LUWI", "PMBI", "PPI", "SANI",
    "TOLI", "TNTI", "JAGI", "PLAI", "UGM", "BOAB", "FAU", "MSI", "BKB"
]

# PC kamu: Core i5-3570 → 4 worker optimal
N_WORKERS = 4

client_iris = Client("IRIS", timeout=30)


In [ ]:
# ============================
# Haversine distance
# ============================
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return 2 * R * atan2(sqrt(a), sqrt(1 - a))

# ============================
# Cache stasiun aktif (1x call)
# ============================
@lru_cache(maxsize=1)
def get_cached_inventory():
    inv = client_iris.get_stations(
        network="*",
        station=",".join(STATION_CODES),
        level="channel"
    )
    return inv

def get_valid_stations_from_cache():
    inv = get_cached_inventory()
    valid_stations = []

    for net in inv:
        for sta in net:
            channels = [c.code for c in sta]
            broadband = [c for c in channels if c[:2] in ["BH", "HH", "EH"]]

            comps = {c[-1] for c in broadband}
            if {"Z", "N", "E"}.issubset(comps):
                valid_stations.append({
                    "station": sta.code,
                    "network": net.code,
                    "latitude": sta.latitude,
                    "longitude": sta.longitude,
                    "elevation": sta.elevation,
                    "channels": list(sorted(set(broadband)))
                })

    return [s for s in valid_stations if s["station"] in STATION_CODES]

# ============================
# Worker per event
# ============================
def process_event(event, valid_stations, processed_ids):
    event_time = event["time_utc"]
    magnitude = event["magnitude"]

    event_time_str = event_time.strftime("%Y%m%d_%H%M%S")

    if any(event_time_str in eid for eid in processed_ids):
        return None

    station_codes = sorted([s["station"] for s in valid_stations])
    station_str = "_".join(station_codes)
    mag_str = str(magnitude).replace(".", "")

    event_id = f"{event_time_str}_M{mag_str}_{station_str}"

    if event_id in processed_ids:
        return None

    return {
        "Event_ID": event_id,
        "Time_UTC": str(event_time),
        "Magnitude": magnitude,
        "Stations": valid_stations
    }

# ============================
# MAIN BUILDER
# ============================
def build_radar_json_promax():
    print("📖 Membaca katalog BMKG & USGS...")

    df_bmkg = pd.read_csv(BMKG_PATH)
    df_usgs = pd.read_csv(USGS_PATH)

    df_bmkg["dt"] = pd.to_datetime(df_bmkg["Date"] + " " + df_bmkg["Time (UTC)"], dayfirst=True, utc=True)
    df_usgs["dt"] = pd.to_datetime(df_usgs["time"], utc=True)

    df_bmkg = df_bmkg[(df_bmkg["dt"].dt.year >= 2001) & (df_bmkg["dt"].dt.year <= 2024)]
    df_usgs = df_usgs[(df_usgs["dt"].dt.year >= 2001) & (df_usgs["dt"].dt.year <= 2024)]

    radar_results = []
    processed_ids = set()

    if os.path.exists(OUTPUT_JSON):
        try:
            with open(OUTPUT_JSON, "r") as f:
                radar_results = json.load(f)
                processed_ids = {entry["Event_ID"] for entry in radar_results}
                print(f"🔄 Resume aktif — {len(processed_ids)} event sudah ada.")
        except:
            print("⚠️ File JSON lama korup, memulai dari awal.")

    print("📡 Pre-cache stasiun 3-komponen dari IRIS...")
    valid_stations = get_valid_stations_from_cache()
    print(f"✅ {len(valid_stations)} stasiun 3C siap dipakai.")

    print("🔍 Mencocokkan BMKG ↔ USGS...")
    matched_events = []

    for _, row in tqdm(df_bmkg.iterrows(), total=len(df_bmkg)):
        bmkg_time = row["dt"]
        bmkg_lat = row["Latitude"]
        bmkg_lon = row["Longitude"]

        df_usgs_near = df_usgs[
            (df_usgs["dt"] >= bmkg_time - pd.Timedelta(seconds=120)) &
            (df_usgs["dt"] <= bmkg_time + pd.Timedelta(seconds=120))
        ]

        if df_usgs_near.empty:
            continue

        df_usgs_near = df_usgs_near.copy()
        df_usgs_near["dist"] = df_usgs_near.apply(
            lambda r: haversine(bmkg_lat, bmkg_lon, r["latitude"], r["longitude"]),
            axis=1
        )

        usgs_match = df_usgs_near.sort_values("dist").iloc[0]
        if usgs_match["dist"] > 50:
            continue

        event_time = UTCDateTime(usgs_match["dt"].to_pydatetime())
        magnitude = usgs_match["mag"]

        matched_events.append({
            "time_utc": event_time,
            "magnitude": magnitude
        })

    print(f"✅ Total event match: {len(matched_events)}")
    print("🚀 Multi-thread processing...")

    new_results = []
    with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
        futures = [
            executor.submit(process_event, ev, valid_stations, processed_ids)
            for ev in matched_events
        ]

        for fut in tqdm(as_completed(futures), total=len(futures)):
            res = fut.result()
            if res is not None:
                new_results.append(res)
                radar_results.append(res)

                with open(OUTPUT_JSON, "w") as f:
                    json.dump(radar_results, f, indent=4)

    print(f"\n🎉 PRO MAX selesai!")
    print(f"➕ Event baru: {len(new_results)}")
    print(f"📁 Lokasi: {OUTPUT_JSON}")


In [ ]:
build_radar_json_promax()


In [ ]:
with open(OUTPUT_JSON) as f:
    data = json.load(f)

df_radar = pd.DataFrame(data)
df_radar.head()
